In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score
from xgboost import XGBClassifier
import joblib  # For saving models
import time  # For measuring computational time
import shutil  # For creating a zip archive

# Load the dataset
data = pd.read_csv("Synthetic Frailty Data.csv")  # Replace with your dataset's file path
data.head()

# Preprocess the data
label_encoder = LabelEncoder()

# Map 'Walking Speed Result' to numeric values
data['Walking Speed Result'] = label_encoder.fit_transform(data['Walking Speed Result'])

# Define input features (X) and output target (y)
X = data[['Walking Speed Test (s)']]
y = data['Walking Speed Result']

# Standardize features for models like SVM and KNN
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Save the scaler and label encoder
joblib.dump(scaler, 'Scaler_WS.pkl')
print("Scaler saved as 'Scaler_WS.pkl'")
joblib.dump(label_encoder, 'Label_Encoder_WS.pkl')  # Save the LabelEncoder
print("Label Encoder saved as 'Label_Encoder_WS.pkl'")

# Initialize models
models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Support Vector Machine": SVC(random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "XGBoost": XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
}

# Train and evaluate models
results = {}
computation_costs = {}

for model_name, model in models.items():
    start_time = time.time()  # Start timing
    model.fit(X_train, y_train)
    end_time = time.time()  # End timing

    # Predictions
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    # Computational cost (training time in seconds)
    training_time = end_time - start_time
    computation_costs[model_name] = training_time

    # Print model results
    print(f"Model: {model_name}")
    print(f"Accuracy: {acc:.2f}")
    print(f"Training Time (seconds): {training_time:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=[str(cls) for cls in label_encoder.classes_]))

    print("-" * 50)
    results[model_name] = acc

    # Save the model as a .pkl file with _WS suffix
    file_name = f"{model_name.replace(' ', '_')}_WS.pkl"  # Append _WS to the file name
    joblib.dump(model, file_name)
    print(f"Model '{model_name}' saved as '{file_name}'")


Scaler saved as 'Scaler_WS.pkl'
Label Encoder saved as 'Label_Encoder_WS.pkl'
Model: Logistic Regression
Accuracy: 0.99
Training Time (seconds): 0.0224

Classification Report:
                   precision    recall  f1-score   support

  Crosswalk Issue       1.00      0.97      0.98        33
High Risk of Fall       0.99      1.00      1.00       117
           Normal       1.00      1.00      1.00        50

         accuracy                           0.99       200
        macro avg       1.00      0.99      0.99       200
     weighted avg       1.00      0.99      0.99       200

--------------------------------------------------
Model 'Logistic Regression' saved as 'Logistic_Regression_WS.pkl'
Model: Random Forest
Accuracy: 1.00
Training Time (seconds): 0.1659

Classification Report:
                   precision    recall  f1-score   support

  Crosswalk Issue       1.00      1.00      1.00        33
High Risk of Fall       1.00      1.00      1.00       117
           Normal    

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [12:36:15] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


In [ ]:
import zipfile

# List of model files to be included in the zip archive
files_to_download = ["Logistic_Regression_WS.pkl", "Random_Forest_WS.pkl", "Support_Vector_Machine_WS.pkl",
                     "K-Nearest_Neighbors_WS.pkl", "XGBoost_WS.pkl", "Scaler_WS.pkl", "Label_Encoder_WS.pkl"]

# Create a zip file and add the files
with zipfile.ZipFile('WS_Models.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file in files_to_download:
        zipf.write(file)

print("Download the 'WS_Models.zip' file from your directory.")


Download the 'WS_Models.zip' file from your directory.
